In [ ]:
# (i) Mount Drive and create the step-5 output folder.
import os
from google.colab import drive

drive.mount("/content/drive")
STEP5 = "/content/drive/MyDrive/laya-vision/step5"
os.makedirs(STEP5, exist_ok=True)
print("Step-5 folder:", STEP5)

In [ ]:
# (ii) Clone this repo (GH_TOKEN from Colab secrets) and install dependencies.
import os, subprocess
from google.colab import userdata

REPO = "/content/laya-vision"
if not os.path.exists(REPO):
    gh_token = userdata.get("GH_TOKEN")
    subprocess.run(
        ["git", "clone", "-q", f"https://{gh_token}@github.com/savita10/laya-vision.git", REPO],
        check=True,
    )
    # Drop the token from the clone's remote config.
    subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin",
                    "https://github.com/savita10/laya-vision.git"], check=True)
COMMIT = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"],
                        capture_output=True, text=True, check=True).stdout.strip()
print("Repo commit (log this in experiments.md):", COMMIT)

!pip install -q "laya==0.3.24" "transformers>=4.48.0" "datasets>=3.0.0" safetensors huggingface_hub numpy pandas tabulate

In [ ]:
# (iii) Fine-tune with upstream's script via the cuda wrapper. Effective batch 64, 4 epochs.
import json, os, subprocess, sys
import torch
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA device. Runtime -> Change runtime type -> select a T4, L4 or A100 GPU.")
gpu = torch.cuda.get_device_name(0)
if "T4" in gpu:
    MICRO_BATCH, GRAD_ACCUM = 4, 16
elif "L4" in gpu or "A100" in gpu:
    MICRO_BATCH, GRAD_ACCUM = 8, 8
else:
    raise RuntimeError(f"GPU {gpu!r} has no agreed batch config (T4: 4x16, L4/A100: 8x8).")
print(f"GPU: {gpu} | micro_batch={MICRO_BATCH} grad_accum={GRAD_ACCUM} (effective {MICRO_BATCH * GRAD_ACCUM})")

BASE_DIR = f"{STEP5}/laya_base"
ITEMS = f"{STEP5}/train_items.pt"
OUT_DIR = f"{STEP5}/laya_finetuned_typed_decisions"

final_cfg = os.path.join(OUT_DIR, "rl_agent_config.json")
if os.path.exists(final_cfg) and json.load(open(final_cfg)).get("fine_tuned"):
    print(f"Final model already in {OUT_DIR}; skipping training.")
else:
    cmd = [
        sys.executable, "-u", f"{REPO}/scripts/run_upstream_finetune.py",
        "--device", "cuda",
        "--epochs", "4",
        "--micro-batch", str(MICRO_BATCH),
        "--grad-accum", str(GRAD_ACCUM),
        "--model-dir", BASE_DIR,
        "--items", ITEMS,
        "--output-dir", OUT_DIR,
    ]
    print(" ".join(cmd))
    with open(f"{STEP5}/train.log", "a") as log:
        log.write(f"# commit {COMMIT} | gpu {gpu} | {' '.join(cmd)}\n")
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in proc.stdout:
            print(line, end="")
            log.write(line)
            log.flush()
        if proc.wait() != 0:
            raise RuntimeError(f"Training failed (exit {proc.returncode}); see {STEP5}/train.log")

In [ ]:
# (iv) Evaluate base and fine-tuned checkpoints on the upstream test split; compare with published.
import json, subprocess, sys
import pandas as pd

runs = {
    "laya (base) - ours": (BASE_DIR, f"{STEP5}/metrics_base.json"),
    "laya-typed-decisions - ours": (OUT_DIR, f"{STEP5}/metrics_finetuned.json"),
}
for name, (model_dir, out) in runs.items():
    print(f"=== {name}: {model_dir}")
    subprocess.run([sys.executable, "-u", f"{REPO}/scripts/eval_upstream.py",
                    "--model-dir", model_dir, "--out", out, "--device", "cuda"], check=True)

# Published: huggingface.co/convaiinnovations/laya model card, "typed-decisions,
# measured on all three checkpoints" (400 cases, 2,000 decisions).
PUBLISHED = {
    "laya (base) - published": {"accuracy": 0.362, "soft_accuracy": 0.332, "brier_score": 0.316, "ece": 0.175, "score_mae": 0.694},
    "laya-typed-decisions - published": {"accuracy": 0.766, "soft_accuracy": 0.471, "brier_score": 0.062, "ece": 0.213, "score_mae": 0.242},
    "TypeSafe Jev 1.13.0 - published": {"accuracy": 0.727, "soft_accuracy": 0.580, "brier_score": 0.148, "ece": 0.144, "score_mae": 0.391},
}
COLS = ["accuracy", "soft_accuracy", "brier_score", "ece", "score_mae", "within_1_level", "latency_p50_ms"]

rows = {}
for name, (_, out) in runs.items():
    report = json.load(open(out))
    rows[name] = report["metrics"]
    print(f"\n{name} per-workflow:", json.dumps(report["per_workflow"], indent=2))
rows.update(PUBLISHED)
order = ["laya (base) - ours", "laya (base) - published",
         "laya-typed-decisions - ours", "laya-typed-decisions - published",
         "TypeSafe Jev 1.13.0 - published"]
table = pd.DataFrame.from_dict(rows, orient="index").reindex(order)[COLS]
print()
print(table.to_markdown(floatfmt=".3f"))